# Speech Emotion Recognition with a pretrained speech model

Instead of hand-made audio features, this notebook uses **WavLM**, an open speech model from Microsoft that was pretrained on 94,000 hours of speech. We feed each clip through it, take its internal representation, and train a small, simple classifier on top. This is called *transfer learning*.

The data and the **speaker-independent split** work the same way as in the first notebook: speakers used for testing are never seen in training.

**To run:** *Runtime > Change runtime type > T4 GPU*, then *Runtime > Run all*. It takes about 20 to 40 minutes. A zip file downloads at the end.

## 1. Setup

In [ ]:
!pip -q install kagglehub librosa soundfile transformers

In [ ]:
import json, os, random, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch
import librosa
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from transformers import AutoFeatureExtractor, AutoModel

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

MODEL_NAME = "microsoft/wavlm-base-plus"   # an alternative to try: "facebook/wav2vec2-base"
SAMPLE_RATE = 16000                        # the rate this model expects
MAX_SECONDS = 6                            # longer clips are cut
OUT = "/content/artifacts_pretrained"
os.makedirs(OUT, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 2. Dataset helpers
The code that reads labels and splits by speaker, written to a file so the repository can reuse it.

In [ ]:
%%writefile ser_data.py
"""Dataset helpers: read emotion and speaker from file names, and split by speaker."""
import os

import numpy as np
import pandas as pd

EMOTIONS = ["angry", "calm", "disgust", "fear", "happy", "neutral", "sad", "surprise"]
EMOTION_TO_ID = {e: i for i, e in enumerate(EMOTIONS)}

RAVDESS_CODES = {"01": "neutral", "02": "calm", "03": "happy", "04": "sad",
                 "05": "angry", "06": "fear", "07": "disgust", "08": "surprise"}
CREMA_CODES = {"ANG": "angry", "DIS": "disgust", "FEA": "fear",
               "HAP": "happy", "NEU": "neutral", "SAD": "sad"}
TESS_CODES = {"angry": "angry", "disgust": "disgust", "fear": "fear", "happy": "happy",
              "neutral": "neutral", "sad": "sad", "ps": "surprise"}
SAVEE_CODES = {"a": "angry", "d": "disgust", "f": "fear", "h": "happy",
               "n": "neutral", "sa": "sad", "su": "surprise"}


def parse_file(dataset, path):
    """Return (emotion, speaker) for one file, or None if the name is not recognised."""
    stem = os.path.splitext(os.path.basename(path))[0]
    if dataset == "ravdess":            # 03-01-05-01-02-01-12.wav (emotion = 3rd, actor = last)
        parts = stem.split("-")
        if len(parts) != 7 or parts[2] not in RAVDESS_CODES:
            return None
        return RAVDESS_CODES[parts[2]], "ravdess_" + parts[6]
    if dataset == "crema":              # 1001_DFA_ANG_XX.wav
        parts = stem.split("_")
        if len(parts) < 3 or parts[2] not in CREMA_CODES:
            return None
        return CREMA_CODES[parts[2]], "crema_" + parts[0]
    if dataset == "tess":               # OAF_back_angry.wav
        parts = stem.split("_")
        label = parts[-1].lower()
        if label not in TESS_CODES:
            return None
        return TESS_CODES[label], "tess_" + parts[0].upper()
    if dataset == "savee":              # DC_sa01.wav
        if "_" not in stem:
            return None
        speaker, rest = stem.split("_", 1)
        label = rest.rstrip("0123456789")
        if label not in SAVEE_CODES:
            return None
        return SAVEE_CODES[label], "savee_" + speaker.upper()
    return None


def build_dataframe(paths):
    """paths: {dataset name: folder}. Returns a table with path, dataset, emotion, speaker, label."""
    rows = []
    for dataset, root in paths.items():
        for folder, _, files in os.walk(root):
            for f in files:
                if f.lower().endswith(".wav"):
                    parsed = parse_file(dataset, f)
                    if parsed:
                        rows.append({"path": os.path.join(folder, f), "file": f, "dataset": dataset,
                                     "emotion": parsed[0], "speaker": parsed[1]})
    df = pd.DataFrame(rows)
    # some Kaggle downloads contain the same folder twice: keep one copy of each file name
    df = df.drop_duplicates(subset=["dataset", "file"])
    df = df.sort_values(["dataset", "file"]).reset_index(drop=True)
    df["label"] = df["emotion"].map(EMOTION_TO_ID)
    return df


def speaker_split(df, seed=42, fractions=(0.70, 0.15, 0.15), tries=500):
    """Split into train/val/test by speaker, so no speaker appears in two parts.

    Gives the same result every time for the same seed, whatever order the files were read in.
    Tries seeds until every emotion appears in all three parts.
    """
    speakers = sorted(df["speaker"].unique())
    for s in range(seed, seed + tries):
        order = np.random.RandomState(s).permutation(len(speakers))
        n_train = int(round(fractions[0] * len(speakers)))
        n_val = int(round(fractions[1] * len(speakers)))
        groups = [set(speakers[i] for i in order[:n_train]),
                  set(speakers[i] for i in order[n_train:n_train + n_val]),
                  set(speakers[i] for i in order[n_train + n_val:])]
        parts = [df[df["speaker"].isin(g)] for g in groups]
        if all(p["emotion"].nunique() == len(EMOTIONS) for p in parts):
            assert not (groups[0] & groups[1] or groups[0] & groups[2] or groups[1] & groups[2])
            return parts[0], parts[1], parts[2], s
    raise RuntimeError("no split found with every emotion in all three parts")

In [ ]:
from ser_data import EMOTIONS, build_dataframe, speaker_split

## 3. Download the four datasets

In [ ]:
import kagglehub

paths = {
    "ravdess": kagglehub.dataset_download("uwrfkaggler/ravdess-emotional-speech-audio"),
    "crema":   kagglehub.dataset_download("ejlok1/cremad"),
    "tess":    kagglehub.dataset_download("ejlok1/toronto-emotional-speech-set-tess"),
    "savee":   kagglehub.dataset_download("ejlok1/surrey-audiovisual-expressed-emotion-savee"),
}
df = build_dataframe(paths)
print(len(df), "files,", df["speaker"].nunique(), "speakers")
print(df.groupby(["dataset", "emotion"]).size().unstack(fill_value=0))

## 4. Split by speaker (train 70%, validation 15%, test 15%)

In [ ]:
train_df, val_df, test_df, split_seed = speaker_split(df, seed=SEED)
for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:5s} {len(part):5d} files, {part['speaker'].nunique():3d} speakers")
print("test files per dataset:", test_df["dataset"].value_counts().to_dict())

## 5. Get WavLM's representation of every clip
For each clip WavLM produces one vector per time step in each of its 13 layers. We average over time, so each clip becomes 13 vectors of 768 numbers. Different layers capture different things, and we will test which layer carries emotion best.

In [ ]:
extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
backbone = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()


@torch.no_grad()
def embed(path):
    audio, _ = librosa.load(path, sr=SAMPLE_RATE, duration=MAX_SECONDS)
    return embed_audio(audio)


@torch.no_grad()
def embed_audio(audio):
    inputs = extractor(audio, sampling_rate=SAMPLE_RATE, return_tensors="pt")
    out = backbone(inputs["input_values"].to(device), output_hidden_states=True)
    layers = torch.stack(out.hidden_states).squeeze(1)          # (13, time, 768)
    return layers.mean(dim=1).cpu().numpy().astype(np.float32)  # (13, 768)


def embed_all(part, name):
    t0 = time.time()
    feats = []
    for i, p in enumerate(part["path"]):
        feats.append(embed(p))
        if (i + 1) % 1000 == 0:
            print(f"  {name}: {i + 1}/{len(part)} ({time.time() - t0:.0f}s)")
    return np.stack(feats)


E_train = embed_all(train_df, "train")
E_val = embed_all(val_df, "val")
E_test = embed_all(test_df, "test")
y_train, y_val, y_test = (p["label"].values for p in (train_df, val_df, test_df))
print(E_train.shape, E_val.shape, E_test.shape)

## 6. Which layer carries the emotion?
We train a simple classifier (logistic regression) on each layer separately and compare on the **validation** speakers. The test speakers are not used for any choice.

In [ ]:
def fit_probe(Xtr, ytr, C=1.0):
    mean, scale = Xtr.mean(axis=0), Xtr.std(axis=0) + 1e-8
    clf = LogisticRegression(C=C, max_iter=3000, class_weight="balanced")
    clf.fit((Xtr - mean) / scale, ytr)
    return clf, mean, scale


def score(clf, mean, scale, X, y):
    pred = clf.predict((X - mean) / scale)
    return accuracy_score(y, pred), f1_score(y, pred, average="weighted")


layer_acc = []
for layer in range(E_train.shape[1]):
    clf, mean, scale = fit_probe(E_train[:, layer], y_train, C=0.1)
    acc, f1 = score(clf, mean, scale, E_val[:, layer], y_val)
    layer_acc.append(acc)
    print(f"layer {layer:2d}: validation accuracy {acc:.3f}")

best_layer = int(np.argmax(layer_acc))
print("best layer:", best_layer)
plt.figure(figsize=(6, 3)); plt.plot(layer_acc, marker="o")
plt.xlabel("WavLM layer"); plt.ylabel("validation accuracy"); plt.tight_layout()
plt.savefig(f"{OUT}/layer_accuracy.png", dpi=150); plt.show()

Choose the classifier's strength (`C`) on the validation speakers too.

In [ ]:
results = {}
for C in [0.003, 0.01, 0.03, 0.1, 0.3, 1.0]:
    clf, mean, scale = fit_probe(E_train[:, best_layer], y_train, C=C)
    results[C] = score(clf, mean, scale, E_val[:, best_layer], y_val)[0]
    print(f"C={C}: validation accuracy {results[C]:.3f}")
best_C = max(results, key=results.get)
print("best C:", best_C)

## 7. Final result on the unseen test speakers
These numbers are the honest result. For reference, the first notebook (hand-made features) got 37.6% with a CNN and 40.5% with a Random Forest on a similar speaker-independent split.

In [ ]:
clf, mean, scale = fit_probe(E_train[:, best_layer], y_train, C=best_C)
pred = clf.predict((E_test[:, best_layer] - mean) / scale)
acc = accuracy_score(y_test, pred)
f1 = f1_score(y_test, pred, average="weighted")
print(f"WavLM layer {best_layer} + logistic regression on unseen speakers: accuracy {acc:.3f}, weighted F1 {f1:.3f}")
print()
print(classification_report(y_test, pred, target_names=EMOTIONS, digits=3))

by_dataset = {}
for name in test_df["dataset"].unique():
    mask = (test_df["dataset"] == name).values
    by_dataset[name] = float(accuracy_score(y_test[mask], pred[mask]))
print("accuracy per dataset:", {k: round(v, 3) for k, v in by_dataset.items()})

cm = confusion_matrix(y_test, pred, labels=range(len(EMOTIONS)))
cm_pct = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm_pct, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(EMOTIONS)), EMOTIONS, rotation=45, ha="right")
ax.set_yticks(range(len(EMOTIONS)), EMOTIONS)
for i in range(len(EMOTIONS)):
    for j in range(len(EMOTIONS)):
        ax.text(j, i, f"{cm_pct[i, j]:.2f}", ha="center", va="center",
                color="white" if cm_pct[i, j] > 0.5 else "black", fontsize=8)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("Test speakers (share of each actual emotion)")
fig.colorbar(im); fig.tight_layout()
fig.savefig(f"{OUT}/confusion_matrix.png", dpi=150)
plt.show()

## 8. Save what the server needs
The classifier is just a table of numbers (no pickle file, which is safer). The server loads WavLM, takes the chosen layer, scales it and multiplies by these numbers.

In [ ]:
np.savez(f"{OUT}/probe.npz", coef=clf.coef_.astype(np.float32), intercept=clf.intercept_.astype(np.float32),
         mean=mean.astype(np.float32), scale=scale.astype(np.float32))
json.dump(EMOTIONS, open(f"{OUT}/labels.json", "w"))
metrics = {
    "model": MODEL_NAME, "layer": best_layer, "C": best_C, "split_seed": int(split_seed),
    "train_files": int(len(train_df)), "val_files": int(len(val_df)), "test_files": int(len(test_df)),
    "train_speakers": int(train_df.speaker.nunique()), "val_speakers": int(val_df.speaker.nunique()),
    "test_speakers": int(test_df.speaker.nunique()),
    "test": {"accuracy": float(acc), "weighted_f1": float(f1), "accuracy_by_dataset": by_dataset},
    "validation_accuracy_by_layer": [float(a) for a in layer_acc],
    "per_class": classification_report(y_test, pred, target_names=EMOTIONS, output_dict=True),
}
json.dump(metrics, open(f"{OUT}/metrics.json", "w"), indent=2)

!cd /content && zip -qr emotion_pretrained.zip artifacts_pretrained && ls -la /content/emotion_pretrained.zip
from google.colab import files
files.download("/content/emotion_pretrained.zip")